In [5]:
import folium
import streamlit as st
from streamlit_folium import st_folium
from folium.plugins import MarkerCluster
import pandas as pd
import requests
from io import StringIO

import pandas as pd
import streamlit as st
import pydeck as pdk
from pydeck.types import String
import numpy as np

In [2]:
# connect and read the data
def load_original_data():
    url = 'https://raw.githubusercontent.com/statcom-um/HSHV_pet_reunion/refs/heads/main/anran/Data/final_noduplicates.csv'
    try:
        return pd.read_csv(url)
    except Exception as e:
        st.error(f"Failed to load data from GitHub. Error: {e}")
        return None

# Load the data
data = load_original_data()
data

,...1,Animal #,Species,Primary Breed,Gender,Altered,Intake Date,Intake Subtype,Location Found,Jurisdiction In,...,...20,...21,...22,...23,LocationPlus,address_google,pnt,lon,lat,Duplicates
0,1,A0040552205,Cat,Domestic Longhair,M,Yes,1/12/2019 11:23,Stray with ID,Huron River Dr and Tuttle Hill,Ypsilanti,...,NaN,Ypsilanti,MI,48197.0,Huron River Dr and Tuttle Hill WC-Ypsilanti Tw...,"S Huron River Dr & Tuttle Hill Rd, Ypsilanti C...","-83.5819195, 42.210799",-83.581919,42.210799,0
1,2,A0040554409,Dog,"Retriever, Labrador",F,Yes,1/12/2019 15:06,Stray with ID,Carpenter and Ellsworth,Pittsfield,...,NaN,Ypsilanti,MI,48197.0,"Carpenter / Ellsworth WC-Pittsfield Twp , Mich...","E Ellsworth Rd, Pittsfield Charter Twp, MI, USA","-83.6995447, 42.230374",-83.699545,42.230374,0
2,3,A0033047934,Dog,Terrier,M,Yes,1/13/2019 12:11,Stray with ID,Waters rd and Wagner Rd,Scio,...,NaN,Ann Arbor,MI,48103.0,"Waters rd and Wagner Rd WC-Scio Twp , Michigan","W Waters Rd & S Wagner Rd, Lodi Township, MI 4...","-83.7984686, 42.242459",-83.798469,42.242459,0
3,4,A0035187693,Dog,"Terrier, Jack Russell",M,Yes,1/14/2019 9:03,Stray with ID,Adams and Harriet,Ypsilanti,...,NaN,Ypsilanti,MI,48197.0,"Adams and Harriet WC-Ypsilanti City , Michigan","Harriet St, Ypsilanti, MI 48197, USA","-83.6201156, 42.2335459",-83.620116,42.233546,0
4,5,A0040580759,Cat,Domestic Shorthair,M,Yes,1/16/2019 14:51,Stray without ID,Oakwood and Sherman,Ypsilanti,...,NaN,Ypsilanti,MI,48198.0,"Oakwood/Sherman WC-Ypsilanti City , Michigan","Sherman St & Oakwood St, Ypsilanti, MI 48197, USA","-83.6289739, 42.243699",-83.628974,42.243699,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6419,11335,A0056846624,Cat,Domestic Shorthair,F,Yes,9/9/2024 10:02,Stray with ID,Carpenter rd and Washtenaw Rd,Pittsfield,...,NaN,NaN,NaN,NaN,Carpenter rd and Washtenaw Rd WC-Pittsfield Tw...,"Washtenaw Ave & Carpenter Rd, Pittsfield Chart...","-83.6807369, 42.2540335",-83.680737,42.254033,0
6420,11337,A0056849779,Cat,Domestic Medium Hair,M,Yes,9/9/2024 14:16,Stray without ID,Wilcox and Edwards N Hines Dr,Plymouth,...,NaN,NaN,NaN,NaN,"Wilcox and Edwards N Hines Dr Plymouth City , ...","Edward N Hines Dr, Plymouth Charter Twp, MI 48...","-83.4683398, 42.394534",-83.468340,42.394534,0
6421,11338,A0056851074,Cat,Domestic Shorthair,F,Yes,9/9/2024 16:00,Kitten/Puppy,Joy Rd and Main St,Plymouth,...,NaN,NaN,NaN,NaN,"Joy Rd and Main St Plymouth Twp , Michigan","S Main St & Joy Rd, Plymouth Charter Twp, MI 4...","-83.4691092, 42.3515033",-83.469109,42.351503,0
6422,11339,A0056851629,Cat,Domestic Shorthair,M,Yes,9/9/2024 16:52,Stray with ID,Carpenter and Washtenaw Ave,Pittsfield,...,NaN,NaN,NaN,NaN,Carpenter and Washtenaw Ave WC-Pittsfield Twp ...,"Washtenaw Ave & Carpenter Rd, Pittsfield Chart...","-83.6807369, 42.2540335",-83.680737,42.254033,0


In [ ]:
data['Species_new'] = data['Species'].apply(lambda x: 'Cat' if x == 'Cat' else ('Dog' if x == 'Dog' else 'Others'))

# Marker colors for each species
marker_colors = {
    'Cat': 'orange',
    'Dog': 'blue',
    'Others': 'green',
}
center_lat = data['lat'].mean()
center_lon = data['lon'].mean()


m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
mCluster_cat = MarkerCluster(name='Cat').add_to(m)
mCluster_dog = MarkerCluster(name='Dog').add_to(m)
mCluster_others = MarkerCluster(name='Others').add_to(m)
# Function to get color based on species
def get_marker_color(species):
    return marker_colors.get(species, 'gray')  # Default to gray if species not found

# Add custom markers for each row in the GeoDataFrame
for idx, row in data.iterrows():
    species = row['Species_new']  # Get the species
    lat = row['lat']  # Get latitude
    lon = row['lon']  # Get longitude
    
    # Create a marker for each row, using the appropriate color
    marker = folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color=get_marker_color(species)),  # Set marker color based on species
        popup=f"Species: {species}<br>Outcome: {row['Outcome Type']}<br>Gender: {row['Gender']}",  # Example of popup info
        tooltip=f"{species} - {row['Outcome Type']}"  # Example tooltip info
    )
    if species == 'Cat':
        mCluster_cat.add_child(marker)
    elif species == 'Dog':
        mCluster_dog.add_child(marker)
    elif species == 'Others':
        mCluster_others.add_child(marker) 

folium.LayerControl().add_to(m)
    # call to render Folium map in Streamlit
st_data = st_folium(m, width=725)


2025-02-05 13:21:13.783 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-05 13:21:13.783 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-05 13:21:13.785 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-05 13:21:13.791 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [51]:
# create found variable
data['Returned'] = np.where(data['Outcome Type'].str.contains('Stray Reclaim'),1,0)

# Subset the data to dogs only
dogs = data[data['Species'] == 'Dog']

# Calculate overall return rate
return_rate = dogs['Returned'].mean()
return_rate

# Extract year from intake date
dogs['Year'] = pd.to_datetime(dogs['Intake Date']).dt.year

# Perform calculation per year
return_rate_per_year = dogs.groupby('Year')['Returned'].mean()
return_rate_per_year

/var/folders/1_/n_9132js1t9dtvwzt235g1900000gn/T/ipykernel_54805/747381819.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dogs['Year'] = pd.to_datetime(dogs['Intake Date']).dt.year


Year
2019    0.369338
2020    0.464865
2021    0.430693
2022    0.373770
2023    0.344118
2024    0.379032
Name: Returned, dtype: float64

In [55]:
dogs['NotReturned'] = np.where(dogs['Outcome Type'].str.contains('Stray Reclaim'),0,1)

# Set up starting location and zoom level
view_state = pdk.ViewState(
    longitude=-83.6,
    latitude=42.3,
    zoom=8, # Higher zoom level means the map is more zoomed in
    min_zoom=6,
    max_zoom=15,
    pitch=20.5, # Tilt the map
    bearing=-5) # Rotate the map

hex_layer = pdk.Layer(
    'HexagonLayer',
    dogs,
    get_position='[lon, lat]',
    auto_highlight=True,
    elevation_scale=50,
    pickable=True,
    # elevation_range=[0, 3000],
    extruded=True, # Show the data in 3D
    radius=1000, # size of hexagons, in meters. 1600 meters is about 1 mile
    # Restrict the map to only show hexagons whose color is in this range
    # Some hexagons will be 0 or 1 due to low sample size, so let's exclude them
    # I'm choosing a minimum of 0.4 so we can see areas where the not-returned rate is higher than the annual average
    color_domain=[0.4, 0.99],
    opacity=0.5,
    # Change color based on data 'NotReturned' column
    # Higher not-returned rate means the color will be more red
    get_color_weight='NotReturned',
    color_aggregation=String('MEAN'),
    coverage=1)

tooltip_html = {
    'html': '<b>Proportion Not Returned:</b> {colorValue} <br><b>Count:</b> {elevationValue}',
    'style': {
        'color': 'white'
    }
}

r = pdk.Deck(
    layers=[hex_layer],
    initial_view_state=view_state,
    tooltip=tooltip_html
)
r

/var/folders/1_/n_9132js1t9dtvwzt235g1900000gn/T/ipykernel_54805/176354760.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dogs['NotReturned'] = np.where(dogs['Outcome Type'].str.contains('Stray Reclaim'),0,1)


{
  "initialViewState": {
    "bearing": -5,
    "latitude": 42.3,
    "longitude": -83.6,
    "maxZoom": 15,
    "minZoom": 6,
    "pitch": 20.5,
    "zoom": 8
  },
  "layers": [
    {
      "@@type": "HexagonLayer",
      "autoHighlight": true,
      "colorAggregation": "MEAN",
      "colorDomain": [
        0.4,
        0.99
      ],
      "coverage": 1,
      "data": [
        {
          "...1": 2,
          "...12": NaN,
          "...13": "Ypsilanti",
          "...14": "MI",
          "...15": 48197.0,
          "...20": NaN,
          "...21": "Ypsilanti",
          "...22": "MI",
          "...23": 48197.0,
          "Altered": "Yes",
          "Animal #": "A0040554409",
          "Duplicates": 0,
          "Finder's Address": "4305 Silverleaf Drive",
          "Gender": "F",
          "Intake Date": "1/12/2019 15:06",
          "Intake Subtype": "Stray with ID",
          "Jurisdiction In": "Pittsfield",
          "Jurisdiction Out": "WC-Pittsfield Twp",
          "Location Found": "Carpenter  and  Ellsworth",
          "LocationPlus": "Carpenter / Ellsworth WC-Pittsfield Twp , Michigan",
          "NotReturned": 0,
          "Outcome Date": "1/13/2019 9:24",
          "Outcome Type": "Stray Reclaim-Owner Visit/Id",
          "Primary Breed": "Retriever, Labrador",
          "Returned": 1,
          "Returned to Address": "4653 Cherry Blosson Lane",
          "Species": "Dog",
          "Year": 2019,
          "address_google": "E Ellsworth Rd, Pittsfield Charter Twp, MI, USA",
          "lat": 42.230374,
          "lon": -83.6995447,
          "pnt": "-83.6995447, 42.230374"
        },
        {
          "...1": 3,
          "...12": NaN,
          "...13": "West Bloomfield",
          "...14": "MI",
          "...15": 48322.0,
          "...20": NaN,
          "...21": "Ann Arbor",
          "...22": "MI",
          "...23": 48103.0,
          "Altered": "Yes",
          "Animal #": "A0033047934",
          "Duplicates": 0,
          "Finder's Address": "6280 Branford Drive",
          "Gender": "M",
          "Intake Date": "1/13/2019 12:11",
          "Intake Subtype": "Stray with ID",
          "Jurisdiction In": "Scio",
          "Jurisdiction Out": "WC-Lodi Twp",
          "Location Found": "Waters rd and Wagner Rd",
          "LocationPlus": "Waters rd and Wagner Rd WC-Scio Twp , Michigan",
          "NotReturned": 0,
          "Outcome Date": "1/13/2019 13:30",
          "Outcome Type": "Stray Reclaim-Microchip",
          "Primary Breed": "Terrier",
          "Returned": 1,
          "Returned to Address": "4805 Diuble Road",
          "Species": "Dog",
          "Year": 2019,
          "address_google": "W Waters Rd & S Wagner Rd, Lodi Township, MI 48103, USA",
          "lat": 42.242459,
          "lon": -83.7984686,
          "pnt": "-83.7984686, 42.242459"
        },
        {
          "...1": 4,
          "...12": NaN,
          "...13": "Ann Arbor",
          "...14": "MI",
          "...15": 48104.0,
          "...20": NaN,
          "...21": "Ypsilanti",
          "...22": "MI",
          "...23": 48197.0,
          "Altered": "Yes",
          "Animal #": "A0035187693",
          "Duplicates": 0,
          "Finder's Address": "2643 Maplewood Avenue",
          "Gender": "M",
          "Intake Date": "1/14/2019 9:03",
          "Intake Subtype": "Stray with ID",
          "Jurisdiction In": "Ypsilanti",
          "Jurisdiction Out": "WC-Ypsilanti City",
          "Location Found": "Adams and Harriet",
          "LocationPlus": "Adams and Harriet WC-Ypsilanti City , Michigan",
          "NotReturned": 0,
          "Outcome Date": "1/14/2019 12:09",
          "Outcome Type": "Stray Reclaim-Lost Report",
          "Primary Breed": "Terrier, Jack Russell",
          "Returned": 1,
          "Returned to Address": "204 S Adams Street",
          "Species": "Dog",
          "Year": 2019,
          "address_google": "Harriet St, Ypsilanti, MI 48197, USA",
          "lat": 42.2335459,
          "lon": -83